In [1]:
import random,os
import pandas as pd
import numpy as np
import missingno as msno
from Data_cleaning_utils import perform_data_cleaning
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,KNNImputer,IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn import set_config
set_config(transform_output='pandas')
import optuna as optuna


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed = 0
seed_everything(seed)

import dagshub
dagshub.init(repo_owner='shapniljoy', repo_name='delivery-time-prediction', mlflow=True)

import mlflow
mlflow.set_tracking_uri("https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow")
mlflow.set_experiment('Lightboost Hyperparameter Tuning')

Accessing as shapniljoy

Initialized MLflow to track repo "shapniljoy/delivery-time-prediction"

Repository shapniljoy/delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/2f629445f6af4f39abb6d2b7815fcd09', creation_time=1782068657306, experiment_id='5', last_update_time=1782068657306, lifecycle_stage='active', name='Lightboost Hyperparameter Tuning', tags={}, trace_location=None, workspace='default'>

# Optuna After dropping Missing values

In [2]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)
df = df.dropna()

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','age',
        'ratings','distance_km','pickup_time','day','month'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (30156, 15) and (30156,) 

Testing data: (7539, 15) and (7539,) 



In [3]:
rating_order = ['less than 4','4-4.5','4.5-5']

distance_order = ['short','medium','long','very_long']

city_type_order = ['Semi-Urban','Urban','Metropolitan']

pickup_time_order = ['5 minutes','10 minutes','15 minutes']

ordinal_encoding = OrdinalEncoder(categories=[rating_order,distance_order,city_type_order,pickup_time_order],
                                  handle_unknown='use_encoded_value', unknown_value=-999)


preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['ratings_cat','distance_km_cat','city_type','pickup_time_cat']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'age_cat','weather','traffic','festival','time_of_day']),
        ],remainder='passthrough')

In [6]:
def objective(trial):

    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}"):

        params = {
        'objective': 'regression',       
        'metric': 'mae',                 
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 100, 3000),          
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 256),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 300),
                
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1.0),

        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10.0, log=True),
        }

        boosting_type = trial.suggest_categorical('boosting_type', ['gbdt', 'dart', 'goss'])
        
        if boosting_type != 'goss':

            params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
            params['subsample_freq'] = 1


        
        model = LGBMRegressor(**params,boosting_type=boosting_type)


        # model = TransformedTargetRegressor(regressor=base_model,func=np.log1p, inverse_func=np.expm1) 

        model = Pipeline([
            ('preprocessor',preprocessor),
            ('model',model)
        ])

        cv_score = cross_validate(model,x_train,y_train,scoring='neg_mean_absolute_error',
                                 cv=KFold(n_splits=5,shuffle=True,random_state=seed),n_jobs=-1,verbose=False,return_train_score=True)

        trial.set_user_attr('train_mae',np.mean(cv_score['train_score']))
        trial.set_user_attr('val_mae',np.mean(cv_score['test_score']))

        mlflow.log_metric('train_mae_score',np.mean(cv_score['train_score']))
        mlflow.log_metric('val_mae_score',np.mean(cv_score['test_score']))
        mlflow.log_param('model',model)

        return np.mean(cv_score['test_score'])



study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=seed),
                            study_name='LGBM Hyperparameter Tuning')

with mlflow.start_run(run_name='Best MAE_2') as parent :
    study.optimize(objective,n_trials=30,show_progress_bar=True)

    mlflow.log_metric('Best MAE_2',study.best_value)
    mlflow.log_params(study.best_params)

    print('Best parameters:', study.best_params)
    print('Best score:', study.best_value)

    

[I 2026-06-22 11:51:32,086] A new study created in memory with name: LGBM Hyperparameter Tuning


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run trial_0 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/5/runs/9f2eaab842284a9f8e78af2701ff3977
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/5
[I 2026-06-22 11:51:52,264] Trial 0 finished with value: -3.8014771246271244 and parameters: {'n_estimators': 1692, 'max_depth': 10, 'num_leaves': 162, 'min_child_samples': 168, 'colsample_bytree': 0.4812893194050143, 'learning_rate': 0.12512806369135343, 'reg_alpha': 0.0562793204741517, 'reg_lambda': 3.6905577292137624, 'boosting_type': 'gbdt', 'subsample': 0.7644474598764522}. Best is trial 0 with value: -3.8014771246271244.
🏃 View run trial_1 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/5/runs/025554a4679744a7ba3d874d0c70111c
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/5
[I 2026-06-22 11:52:03,319] Trial 1 finished with value: -3.986810588063637 and paramet

In [ ]:
study.trials_dataframe()[['user_attrs_train_mae','user_attrs_val_mae']].sort_values(by='user_attrs_val_mae',ascending=False)

,user_attrs_train_mae,user_attrs_val_mae
25,-3.378260,-3.639431
24,-3.318962,-3.644048
23,-3.383500,-3.650129
22,-3.390764,-3.650535
26,-3.233793,-3.652710
21,-3.411781,-3.653075
20,-3.298178,-3.664369
11,-3.306595,-3.669893
27,-3.351595,-3.674083
18,-3.396404,-3.674602


In [7]:
study.best_params

{'n_estimators': 1940,
 'max_depth': 7,
 'num_leaves': 57,
 'min_child_samples': 228,
 'colsample_bytree': 0.8452545147323064,
 'learning_rate': 0.046640660322683554,
 'reg_alpha': 0.020341454480246005,
 'reg_lambda': 0.10776961230711216,
 'boosting_type': 'dart',
 'subsample': 0.9481361810641186}

In [8]:
best_model = LGBMRegressor(**study.best_params,
                           objective= 'regression',       
                            metric= 'mae',                 
                            random_state = seed)

final_model_pipe = Pipeline([
    ('preprocessor',preprocessor),
    ('model',best_model)
])
final_lgbm_model = final_model_pipe.fit(x_train,y_train)

# final_cat_model =TransformedTargetRegressor(regressor=best_model,func=np.log1p, inverse_func=np.expm1).fit(x_train,y_train)

y_pred_train = final_lgbm_model.predict(x_train)
y_pred_test = final_lgbm_model.predict(x_test)

print(f"Training error: {mean_absolute_error(y_train,y_pred_train)}")
print(f"Testing error: {mean_absolute_error(y_test,y_pred_test)}")
print(f"Training R2 score: {r2_score(y_train,y_pred_train)}")
print(f"Testing R2 score: {r2_score(y_test,y_pred_test)}")


mlflow.log_metric('Training error',mean_absolute_error(y_train,y_pred_train))
mlflow.log_metric('Testing error',mean_absolute_error(y_test,y_pred_test))
mlflow.log_metric('Training R2 score',r2_score(y_train,y_pred_train))
mlflow.log_metric('Testing R2 score',r2_score(y_test,y_pred_test))

mlflow.sklearn.log_model(best_model,name='LGBM Model_3')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000690 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 70
[LightGBM] [Info] Number of data points in the train set: 30156, number of used features: 31
[LightGBM] [Info] Start training from score 26.649755
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

2026/06/22 12:06:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
